# Early Exit Inference with BERT and RoBERTa

This notebook implements and evaluates **Early Exit** strategies for Transformer-based models (BERT and RoBERTa). Early exiting optimizes inference by allowing the model to terminate the forward pass early if it reaches a high level of confidence at an intermediate layer.

### Key Components:
1. **`EarlyExitMixin`**: A shared infrastructure providing tracking for active samples, confidence evaluation (via Softmax), and dynamic batch shrinking.
2. **Model Adapters**: Custom wrappers for `BertTiny`, `BertBase`, and `RoBERTa` that integrate intermediate classification routing.
3. **Dynamic Shrinking**: During inference, samples that meet the confidence threshold are "exited," and the remaining batch is compressed to save computation in subsequent layers.
4. **Benchmarking**: Utilities to measure accuracy trade-offs and inference latency across various thresholds.


In [ ]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer, 
    DataCollatorWithPadding
)
from torch.utils.data import DataLoader
from chop import MaseGraph
from chop.tools import get_tokenized_dataset


In [8]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    print("Using CUDA device:", props.name)
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available — using CPU")

checkpoint = "DeepWokLab/bert-tiny"
tokenizer_checkpoint = "DeepWokLab/bert-tiny"
dataset_name = "imdb"

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)


INFO     Tokenizing dataset imdb with AutoTokenizer for DeepWokLab/bert-tiny.


Using CUDA device: NVIDIA GeForce RTX 4080


In [ ]:
@torch.no_grad()
def eval_accuracy(model, dataloader, device="cuda"):
    model.eval()
    correct, total = 0, 0
    for batch in tqdm(dataloader, desc="Eval Accuracy", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        logits = out["logits"] if isinstance(out, dict) else out.logits
        correct += (logits.argmax(dim=-1) == batch["labels"]).sum().item()
        total += batch["labels"].size(0)
    acc = correct / total
    print(f"Accuracy: {acc * 100:.2f}% ({correct}/{total})")
    return acc

@torch.no_grad()
def eval_speed(model, dataloader, device="cuda", num_batches=100, warmup=10):
    model.eval()
    if isinstance(device, str):
        device = torch.device(device)
        
    batches = list(dataloader)[:warmup + num_batches]
    for b in batches[:warmup]:
        model(**{k: v.to(device) for k, v in b.items()})
    
    if device.type == "cuda":
        torch.cuda.synchronize()

    times, samples = [], 0
    for b in batches[warmup:]:
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(**{k: v.to(device) for k, v in b.items()})
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
        samples += b["input_ids"].size(0)
    
    total_time = sum(times)
    avg_per_batch_ms = np.mean(times) * 1000
    avg_per_sample_ms = (total_time / samples) * 1000
    print(f"Throughput: {samples / total_time:.1f} samples/sec | Latency: {avg_per_sample_ms:.2f} ms/sample")
    return [avg_per_batch_ms, avg_per_sample_ms]


In [ ]:
class EarlyExitBertTiny(nn.Module):
    def __init__(self, original_model, threshold=0.9):
        super().__init__()
        self.bert = original_model.bert
        self.classifier = original_model.classifier
        self.threshold = threshold
        self.encoder_layers = nn.ModuleList(list(self.bert.encoder.layer.children()))
        self.num_layers = len(self.encoder_layers)

    def forward(self, input_ids, attention_mask, labels=None):
        device = input_ids.device
        batch_size = input_ids.size(0)
        extended_attention_mask = self.bert.get_extended_attention_mask(attention_mask, input_ids.size(), device)
        hidden_states = self.bert.embeddings(input_ids=input_ids)
        
        final_logits = torch.zeros(batch_size, self.classifier.out_features, device=device)
        active_mask = extended_attention_mask
        active_indices = torch.arange(batch_size, device=device)

        for i, layer_module in enumerate(self.encoder_layers):
            layer_outputs = layer_module(hidden_states, attention_mask=active_mask)
            hidden_states = layer_outputs[0]
            
            cls_token = hidden_states[:, 0]
            pooled_output = self.bert.pooler.dense(cls_token)
            pooled_output = self.bert.pooler.activation(pooled_output)
            logits = self.classifier(pooled_output)
            
            if i == self.num_layers - 1:
                final_logits[active_indices] = logits
                break

            probs = F.softmax(logits, dim=-1)
            max_probs, _ = torch.max(probs, dim=-1)
            confident = max_probs >= self.threshold
            
            final_logits[active_indices[confident]] = logits[confident]
            not_confident = ~confident
            if not not_confident.any():
                break
            
            active_indices = active_indices[not_confident]
            hidden_states = hidden_states[not_confident]
            active_mask = active_mask[not_confident]
        
        return {"logits": final_logits}


In [ ]:
class EarlyExitMixin:
    def init_early_exit(self, original_model, threshold):
        self.threshold = threshold
        self.num_labels = original_model.config.num_labels
        self.pooler = None 
        self.classifier = None 

    def compute_logits(self, hidden_states):
        if self.pooler is not None:
            return self.classifier(self.pooler(hidden_states))
        
        try:
            return self.classifier(hidden_states[:, 0])
        except Exception:
            return self.classifier(hidden_states)

    def evaluate_confidence(self, logits, active_indices, final_logits):
        probs = F.softmax(logits, dim=-1)
        max_p, _ = torch.max(probs, dim=-1)
        confident = max_p >= self.threshold
        
        final_logits[active_indices[confident]] = logits[confident]
        
        return ~confident


In [ ]:
class RobertaEarlyExit(nn.Module, EarlyExitMixin):
    def __init__(self, original_model, threshold=0.75):
        super().__init__()
        self.init_early_exit(original_model, threshold)
        
        self.embeddings = original_model.roberta.embeddings
        self.layers = nn.ModuleList(original_model.roberta.encoder.layer)
        self.classifier = original_model.classifier
        
        self.num_layers = len(self.layers)

    def compute_logits(self, hidden_states):
        return self.classifier(hidden_states)

    def forward(self, input_ids, attention_mask, **kwargs):
        device = input_ids.device
        batch_size = input_ids.size(0)

        hidden_states = self.embeddings(input_ids=input_ids)
        
        extended_attention_mask = attention_mask[:, None, None, :]
        extended_attention_mask = extended_attention_mask.to(dtype=hidden_states.dtype)
        active_mask = (1.0 - extended_attention_mask) * -10000.0

        final_logits = torch.zeros(batch_size, self.num_labels, device=device)
        active_indices = torch.arange(batch_size, device=device)

        for i, layer in enumerate(self.layers):
            layer_outputs = layer(hidden_states, attention_mask=active_mask)
            hidden_states = layer_outputs[0]

            logits = self.compute_logits(hidden_states)

            if i == self.num_layers - 1:
                final_logits[active_indices] = logits
                break

            not_confident = self.evaluate_confidence(logits, active_indices, final_logits)
            if not not_confident.any(): 
                break 
                
            active_indices = active_indices[not_confident]
            hidden_states = hidden_states[not_confident]
            active_mask = active_mask[not_confident]

        return {"logits": final_logits}


In [ ]:
class BertEarlyExit(nn.Module, EarlyExitMixin):
    def __init__(self, original_model, threshold=0.75):
        super().__init__()
        self.init_early_exit(original_model, threshold)
        
        self.embeddings = original_model.bert.embeddings
        self.layers = original_model.bert.encoder.layer
        self.pooler = original_model.bert.pooler
        self.classifier = original_model.classifier
        
        self.base_model = original_model.bert
        self.num_layers = len(self.layers)

    def forward(self, input_ids, attention_mask, **kwargs):
        device = input_ids.device
        batch_size = input_ids.size(0)

        hidden_states = self.embeddings(input_ids=input_ids)
        active_mask = self.base_model.get_extended_attention_mask(attention_mask, input_ids.size(), device)
        
        final_logits = torch.zeros(batch_size, self.num_labels, device=device)
        active_indices = torch.arange(batch_size, device=device)

        for i, layer in enumerate(self.layers):
            layer_outputs = layer(hidden_states, attention_mask=active_mask)
            hidden_states = layer_outputs[0]

            logits = self.compute_logits(hidden_states)

            if i == self.num_layers - 1:
                final_logits[active_indices] = logits
                break

            not_confident = self.evaluate_confidence(logits, active_indices, final_logits)
            
            if not not_confident.any(): 
                break 
                
            active_indices = active_indices[not_confident]
            hidden_states = hidden_states[not_confident]
            active_mask = active_mask[not_confident]

        return {"logits": final_logits}


## bert tiny

In [6]:
mg_baseline = MaseGraph.from_checkpoint("/vol/bitbucket/ug22/adls-data/models/bert-tiny-imdb-baseline")
baseline_model = mg_baseline.model.to(DEVICE)

# 1. Reconstruct the model (as you already did)
hf_model = AutoModelForSequenceClassification.from_pretrained("DeepWokLab/bert-tiny", num_labels=2)
hf_model.load_state_dict(baseline_model.state_dict(), strict=False)
hf_model = hf_model.to(DEVICE)
print(hf_model.bert)
THRESHOLD = 0.75
print(f"Baseline Hidden Size: {baseline_model.config.hidden_size}")
print(f"HF Model Hidden Size: {hf_model.config.hidden_size}")
print("\n" + "="*50)
print(f"EARLY EXIT (Threshold = {THRESHOLD})")
print("="*50)

# 2. FIX: Call the dedicated BERT Adapter! No more string paths.
early_exit_model = BertEarlyExit(
    original_model=hf_model, 
    threshold=THRESHOLD
).to(DEVICE)

# 3. Now run your evaluation
eval_accuracy(early_exit_model, eval_dataloader, device=DEVICE)
eval_speed(early_exit_model, eval_dataloader, device=DEVICE)

# 4. baseline model for reference
print("\n" + "="*50)
print("BASELINE BERT-TINY (No Early Exit)")
print("="*50)
eval_accuracy(hf_model, eval_dataloader, device=DEVICE)
eval_speed(hf_model, eval_dataloader, device=DEVICE)

WARNING  Node finfo not found in loaded metadata.
WARNING  Node getattr_2 not found in loaded metadata.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepWokLab/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Baseline Hidden Size: 128
HF Model Hidden Size: 128

EARLY EXIT (Threshold = 0.75)


Eval Accuracy:   0%|          | 0/391 [00:00<?, ?it/s]/vol/bitbucket/nr722/adls-project/.venv/lib/python3.11/site-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
Eval Accuracy: 100%|██████████| 391/391 [00:05<00:00, 65.69it/s]


Accuracy: 81.20% (20299/25000)
6400 samples in 0.2945s
Throughput:      21731.6 samples/sec
Avg batch:       2.95 ms
Avg per-sample:  0.05 ms

Loading baseline model...
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-1): 2 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_featur

Eval Accuracy: 100%|██████████| 391/391 [00:06<00:00, 63.33it/s]


Accuracy: 82.34% (20586/25000)
6400 samples in 0.4910s
Throughput:      13035.9 samples/sec
Avg batch:       4.91 ms
Avg per-sample:  0.08 ms


[np.float64(4.909501090023696), 0.07671095453162025]

## Roberta

In [ ]:
ROBERTA_MASE_PATH = "/vol/bitbucket/ug22/adls-data/models/roberta-imdb-baseline"
MODEL_CKPT = "/vol/bitbucket/ug22/adls-data/models/roberta-imdb-baseline"
TOKENIZER_CKPT = "roberta-base"

dataset, tokenizer = get_tokenized_dataset(dataset="imdb", checkpoint=TOKENIZER_CKPT, return_tokenizer=True)
collator = DataCollatorWithPadding(tokenizer)

def make_loader(split, bs, shuffle=False):
    ds = dataset[split].remove_columns(["text"]).rename_column("label", "labels")
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collator)

eval_loader = make_loader("test", 64)


INFO     Tokenizing dataset imdb with AutoTokenizer for roberta-base.


In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

# Ensure DEVICE is correctly typed for eval_speed
DEVICE_OBJ = torch.device(DEVICE) if isinstance(DEVICE, str) else DEVICE

# Load model baseline from MaseGraph
threshold = 0.75
mg = MaseGraph.from_checkpoint(MODEL_CKPT)
mase_model = mg.model.to(DEVICE_OBJ)

# Reconstruct the model using standard HuggingFace structure
# This is required because MaseGraph models (GraphModules) cannot be dynamically 
# sliced for Early Exit without breaking their internal computational graph.
hf_model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
hf_model.load_state_dict(mase_model.state_dict(), strict=False)
hf_model = hf_model.to(DEVICE_OBJ).eval()

model = RobertaEarlyExit(hf_model, threshold=threshold).to(DEVICE_OBJ)

# 5. Run Eval
eval_accuracy(model, eval_loader, device=DEVICE_OBJ)
eval_speed(model, eval_loader, device=DEVICE_OBJ)

# 6. Baseline for reference
print("\n" + "="*50)
print("BASELINE ROBERTA (No Early Exit)")
print("="*50)
eval_accuracy(hf_model, eval_loader, device=DEVICE_OBJ)
eval_speed(hf_model, eval_loader, device=DEVICE_OBJ)

WARNING  Node finfo not found in loaded metadata.
WARNING  Node getattr_2 not found in loaded metadata.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


6400 samples in 25.4852s
Throughput:      251.1 samples/sec
Avg batch:       254.85 ms
Avg per-sample:  3.98 ms

BASELINE ROBERTA (No Early Exit)
6400 samples in 28.1349s
Throughput:      227.5 samples/sec
Avg batch:       281.35 ms
Avg per-sample:  4.40 ms


[np.float64(281.348560149745), 4.396071252339766]

In [ ]:
import matplotlib.pyplot as plt

thresholds = [0.5, 0.65, 0.7, 0.73, 0.75, 0.77, 0.8, 0.9]
accuracies = []
latencies = []

print("Evaluating different confidence thresholds...")
for t in thresholds:
    print(f"\n--- Threshold: {t} ---")
    ee_model = EarlyExitBert(hf_model, threshold=t).to(DEVICE)
    acc = eval_accuracy(ee_model, eval_dataloader, device=DEVICE)
    lat, _ = eval_speed(ee_model, eval_dataloader, device=DEVICE)
    accuracies.append(acc * 100)
    latencies.append(lat)

# Get baseline for comparison
print(f"\n--- Baseline Model ---")
base_acc = eval_accuracy(baseline_model, eval_dataloader, device=DEVICE) * 100
base_lat, _ = eval_speed(baseline_model, eval_dataloader, device=DEVICE)

# Simplified Plotting
plt.figure(figsize=(9, 6))

# Plot Early Exit points
plt.scatter(latencies, accuracies, color='b', s=100, label='Early Exit')
plt.plot(latencies, accuracies, 'b--', alpha=0.5)

for i, t in enumerate(thresholds):
    plt.annotate(f"T={t}", (latencies[i], accuracies[i]), 
                 textcoords="offset points", xytext=(0, 10), ha='center')

# Plot Baseline
plt.scatter([base_lat], [base_acc], color='r', marker='*', s=200, label='Baseline')
plt.annotate("Baseline", (base_lat, base_acc), 
             textcoords="offset points", xytext=(0, -15), ha='center', color='red')

plt.title('Performance vs Efficiency Trade-off: Accuracy vs Latency')
plt.xlabel('Average Inference Latency per batch (ms)')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.show()